# 🧠 Week 6 — Day 4
## Building & Training a Neural Network in Keras
### Phase 3 — Deep Learning & Applied Project | BinX Tech AI & ML Internship

![status](https://img.shields.io/badge/status-complete-brightgreen) ![phase](https://img.shields.io/badge/phase-3%20%7C%20Deep%20Learning-blueviolet) ![sprint](https://img.shields.io/badge/sprint-1%20%7C%20Day%204-informational) ![dataset](https://img.shields.io/badge/dataset-Heart%20Failure-red) ![python](https://img.shields.io/badge/python-3.10%2B-blue) ![libraries](https://img.shields.io/badge/libraries-TensorFlow%20%7C%20Keras%20%7C%20Scikit--learn-orange) ![baseline](https://img.shields.io/badge/baseline%20to%20beat-Logistic%20Regression-darkgreen)

| | |
|---|---|
| **Intern** | Zayan Shawareb |
| **Program** | BinX Tech — AI & Machine Learning Internship |
| **Phase** | Phase 3 — Deep Learning & Applied Project (Sprint 1, Day 4 of 5) |
| **Capstone** | Cardiac Patient Monitoring System |
| **Focus** | Keras Sequential API • Compile/Fit/Evaluate • Batch Normalization & Dropout |
| **Hours** | 8 |

---

> *"You will not implement neural networks from scratch in production — frameworks handle the math. TensorFlow with its Keras API is the standard high-level framework."*
> — BinX Tech, Week 6 Curriculum

---

### 📑 Table of Contents
1. [Setup & Data Prep](#1-setup--data-prep-carried-over-from-day-1)
2. [TensorFlow / Keras](#2-tensorflow--keras)
3. [Building a Model — Sequential API](#3-building-a-model-with-the-sequential-api)
4. [Compile, Train, Evaluate](#4-compile-train-evaluate)
5. [Reading the Training History](#5-reading-the-training-history)
6. [Batch Normalization & Dropout](#6-batch-normalization--dropout)
7. [🔬 Hands-On Lab: Training a Neural Network](#7-hands-on-lab-training-a-neural-network)
8. [Git Workflow](#8-commit--update-the-pull-request)
9. [Key Takeaways & Next Steps](#9-key-takeaways--next-steps)


## 🎯 Learning Objectives

By the end of today I will be able to:

- [ ] Build a neural network with the Keras **Sequential API**.
- [ ] Compile, train, and evaluate the network, and read its **training history**.
- [ ] Apply **batch normalization** and **dropout** to stabilize training and reduce overfitting.
- [ ] Compare the trained network's score against the Day 1 baseline.


## 1. Setup & Data Prep (carried over from Day 1)

Same cleaning pipeline as `week6/day1/day1.ipynb`, kept self-contained here so this notebook runs independently: replace invalid `0` values in `Cholesterol`/`RestingBP` with the median, one-hot encode categoricals, split, and scale.


In [1]:
# 📦 Setup — imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix,
    classification_report, RocCurveDisplay
)

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("✅ Environment ready. TensorFlow version:", tf.__version__)


ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# 📂 Load + clean the Cardiac Patient Monitoring System dataset
DATA_PATH = "../../capstone/heart.csv"   # adjust if your capstone folder lives elsewhere

df = pd.read_csv(DATA_PATH)

df_clean = df.copy()
median_chol = df_clean.loc[df_clean["Cholesterol"] > 0, "Cholesterol"].median()
df_clean["Cholesterol"] = df_clean["Cholesterol"].replace(0, median_chol)
median_bp = df_clean.loc[df_clean["RestingBP"] > 0, "RestingBP"].median()
df_clean["RestingBP"] = df_clean["RestingBP"].replace(0, median_bp)

categorical_cols = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]
df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

X = df_encoded.drop(columns=["HeartDisease"])
y = df_encoded["HeartDisease"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

n_features = X_train_scaled.shape[1]
print(f"Train: {X_train_scaled.shape} | Test: {X_test_scaled.shape} | Features: {n_features}")


### 📌 Baseline to Beat (from Day 1)

> **Model:** Logistic Regression
> **Accuracy:** `____` *(copy the exact value from your `day1.ipynb` run)*
> **ROC-AUC:** `____` *(copy the exact value from your `day1.ipynb` run)*

Every network trained today is scored against this number.


In [ ]:
# ✍️ Record the Day 1 baseline here so this notebook can compare against it later
BASELINE_ACCURACY = 0.0000   # <-- replace with your Day 1 value
BASELINE_ROC_AUC  = 0.0000   # <-- replace with your Day 1 value


## 2. TensorFlow / Keras

Frameworks handle the math — you describe the architecture **layer by layer**, and Keras handles forward propagation, backpropagation, and optimization automatically (via `GradientTape` under the hood). PyTorch is the main alternative on the program's stack, used as needed.


## 3. Building a Model with the Sequential API

The Keras **Sequential API** stacks layers in order. A `Dense` (fully connected) layer connects every input to every neuron — exactly the ANN structure from Day 1. Our output task is **binary classification**, so the output layer is a single `Dense(1, activation="sigmoid")` neuron.


In [ ]:
# 🏗️ Model v1 — plain Sequential network, no regularization yet
model_v1 = Sequential([
    Dense(64, activation="relu", input_shape=(n_features,)),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid"),   # binary output
], name="cardiac_nn_v1")

model_v1.summary()


## 4. Compile, Train, Evaluate

The three-step Keras workflow mirrors the Scikit-learn API from Week 3: `compile()` sets the optimizer, loss, and metrics; `fit()` runs the training loop; `evaluate()` measures performance on the test set.


In [ ]:
# ⚙️ Compile & train Model v1
model_v1.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_v1 = model_v1.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    verbose=0
)

print("✅ Training complete.")
test_loss_v1, test_acc_v1 = model_v1.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Model v1 — Test Loss: {test_loss_v1:.4f} | Test Accuracy: {test_acc_v1:.4f}")


In [ ]:
# 📈 Full evaluation of Model v1 (accuracy, ROC-AUC, confusion matrix)
y_proba_v1 = model_v1.predict(X_test_scaled, verbose=0).ravel()
y_pred_v1 = (y_proba_v1 >= 0.5).astype(int)

v1_accuracy = accuracy_score(y_test, y_pred_v1)
v1_roc_auc = roc_auc_score(y_test, y_proba_v1)

print(f"📌 Model v1 Accuracy : {v1_accuracy:.4f}")
print(f"📌 Model v1 ROC-AUC  : {v1_roc_auc:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred_v1, target_names=["No Disease", "Disease"]))


## 5. Reading the Training History

The `history` object records the loss and metric **per epoch**, for both training and validation. Plotting it is how we diagnose overfitting/underfitting — the exact same reasoning from Week 4.


In [ ]:
# 📊 Plot Model v1 training history — loss & accuracy curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(history_v1.history["loss"], label="Train Loss", color="#2563eb")
axes[0].plot(history_v1.history["val_loss"], label="Validation Loss", color="#dc2626")
axes[0].set_title("Model v1 — Loss Curve")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Binary Cross-Entropy Loss")
axes[0].legend()

axes[1].plot(history_v1.history["accuracy"], label="Train Accuracy", color="#2563eb")
axes[1].plot(history_v1.history["val_accuracy"], label="Validation Accuracy", color="#dc2626")
axes[1].set_title("Model v1 — Accuracy Curve")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()


<div style="background-color:#eef6fb; padding:15px; border-radius:8px; border-left:4px solid #2563eb;">
<b>Observation</b><br><br>
The training loss keeps decreasing steadily, but watch the <b>gap between train and validation loss</b> — if validation loss starts rising while train loss keeps falling, that's classic <b>overfitting</b> (the network memorizing rather than generalizing). This is exactly why Section 6 adds Dropout and Batch Normalization: to close that gap before we tune further on Day 5.
</div>


## 6. Batch Normalization & Dropout

Two standard layers improve training and fight overfitting:

- **Batch Normalization** — normalizes each layer's inputs during training, making the network train faster and more stably.
- **Dropout** — randomly "switches off" a fraction of neurons during each training step, forcing the network not to rely too heavily on any one path. The deep-learning equivalent of the regularization from Week 4.


In [ ]:
# 🏗️ Model v2 — with BatchNormalization + Dropout
model_v2 = Sequential([
    Dense(64, activation="relu", input_shape=(n_features,)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation="relu"),
    BatchNormalization(),
    Dropout(0.2),
    Dense(1, activation="sigmoid"),
], name="cardiac_nn_v2_regularized")

model_v2.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model_v2.summary()


In [ ]:
# ⚙️ Train Model v2
history_v2 = model_v2.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    verbose=0
)

y_proba_v2 = model_v2.predict(X_test_scaled, verbose=0).ravel()
y_pred_v2 = (y_proba_v2 >= 0.5).astype(int)

v2_accuracy = accuracy_score(y_test, y_pred_v2)
v2_roc_auc = roc_auc_score(y_test, y_proba_v2)

print(f"📌 Model v2 (regularized) Accuracy : {v2_accuracy:.4f}")
print(f"📌 Model v2 (regularized) ROC-AUC  : {v2_roc_auc:.4f}")


In [ ]:
# 📊 Compare v1 vs v2 loss curves side-by-side
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(history_v1.history["val_loss"], label="v1 — no regularization", color="#dc2626")
axes[0].plot(history_v2.history["val_loss"], label="v2 — BatchNorm + Dropout", color="#16a34a")
axes[0].set_title("Validation Loss — v1 vs v2")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Val Loss")
axes[0].legend()

axes[1].bar(
    ["Baseline\n(LogReg)", "Model v1", "Model v2\n(regularized)"],
    [BASELINE_ACCURACY, v1_accuracy, v2_accuracy],
    color=["#94a3b8", "#dc2626", "#16a34a"]
)
axes[1].set_title("Test Accuracy Comparison")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()


<div style="background-color:#eef6fb; padding:15px; border-radius:8px; border-left:4px solid #16a34a;">
<b>Observation</b><br><br>
Comparing the validation loss curves shows whether BatchNorm + Dropout actually helped — a v2 curve that stays lower and flatter than v1 confirms the regularization is doing its job. The bar chart puts the baseline Logistic Regression side-by-side with both networks: <b>the neural network only earns its place in this project if it clears the baseline bar</b> — remind yourself of that Day 1 baseline value above before drawing conclusions.
</div>


## 7. 🔬 Hands-On Lab: Training a Neural Network

**Steps:**
1. ✅ Build a Keras Sequential network appropriate for the Phase 3 project task (done — Model v1).
2. ✅ Compile with Adam and binary cross-entropy, train with a validation split for 50 epochs.
3. ✅ Plot training vs. validation loss and accuracy, diagnose the fit.
4. ✅ Add Dropout and Batch Normalization, compare the new loss curves to the previous run (Model v2).
5. Evaluate on the test set and compare the score to the Day 1 baseline — fill in the summary table below.

### 📋 Day 4 Results Summary

| Model | Accuracy | ROC-AUC | Beats Baseline? |
|---|---|---|---|
| Baseline (Logistic Regression, Day 1) | `____` | `____` | — |
| Model v1 (plain NN) | `____` | `____` | `Yes / No` |
| Model v2 (BatchNorm + Dropout) | `____` | `____` | `Yes / No` |

Fill in the exact numbers from the cells above before committing — the mentor review will check this table on Day 5's Sprint Review.


## 8. Commit & Update the Pull Request

Run these in your terminal (not in the notebook):

```bash
cd C:\Users\lenovo\binx-ai-ml-internship-
git checkout week6/day1
mkdir week6\day4
git add week6/day4/day4.ipynb week6/day4/README_DAY_4.md
git commit -m "Week 6 Day 4: Keras Sequential model, BatchNorm/Dropout, training history"
git push
```

> 💡 If you're keeping one PR per sprint (recommended), stay on the `week6/day1` branch and just add the new `week6/day4/` folder to it — no need for a new branch each day. If your mentor prefers one branch per day, use `git checkout -b week6/day4` instead.


## 9. Key Takeaways & Next Steps

- ✅ Built a neural network with the Keras **Sequential API** — `Dense` layers stacking exactly like the Day 1 architecture diagram.
- ✅ `compile()` → `fit()` → `evaluate()` mirrors the Scikit-learn workflow from Week 3.
- ✅ The `history` object is how we **diagnose** training — always read the loss curves before trusting a result.
- ✅ **BatchNormalization** stabilizes training; **Dropout** fights overfitting — both are now standard tools.
- ✅ Two trained networks (v1, v2) benchmarked against the Day 1 baseline.

### ➡️ Day 5 Preview
Systematic tuning (one hyperparameter at a time), `EarlyStopping` and `ModelCheckpoint` callbacks, assembling the final Sprint 1 evidence, and running the full **Sprint Review & Retrospective**.
